In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CHOW_FP = r"C:\Repositories\white-bowblis-nhmc\data\interim\chow.csv"
OUT_DIR = r"C:\Repositories\white-bowblis-nhmc\outputs\plots"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(CHOW_FP)

# -----------------------------
# Helpers
# -----------------------------
def bin_0_1_2p(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x <= 0:
        return "0"
    if x == 1:
        return "1"
    return "2+"

def first_valid_date(row, cols):
    """Return first non-missing parsed date across cols (row-wise)."""
    for c in cols:
        if c in row.index:
            d = pd.to_datetime(row[c], errors="coerce")
            if pd.notna(d):
                return d
    return pd.NaT

# -----------------------------
# Cross-tab heatmap (0/1/2+)
# -----------------------------
df["nhc_bin"] = df["n_chow_nh_compare"].apply(bin_0_1_2p)
df["mcr_bin"] = df["n_chow_mcr"].apply(bin_0_1_2p)

ct = pd.crosstab(df["nhc_bin"], df["mcr_bin"]).reindex(
    index=["0","1","2+"], columns=["0","1","2+"], fill_value=0
)

plt.figure()
mat = ct.values
plt.imshow(mat)
plt.xticks(range(ct.shape[1]), ct.columns)
plt.yticks(range(ct.shape[0]), ct.index)
plt.xlabel("HCRIS/MCR changes (0 / 1 / 2+)")
plt.ylabel("NHC changes (0 / 1 / 2+)")
plt.title("Cross-tab of Ownership Changes: NHC vs HCRIS/MCR")

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        plt.text(j, i, f"{mat[i, j]:,}", ha="center", va="center")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "crosstab_nhc_vs_mcr_heatmap.png"), dpi=200)
plt.close()

print("\n=== Cross-tab (0/1/2+) ===")
print(ct)

# -----------------------------
# Validated changes: only facilities with exactly 1 in NHC AND 1 in MCR
# and count changes over study period
# -----------------------------
# Identify date columns
date_cols_nhc = [c for c in df.columns if c.startswith("nh_compare_chow_") and c.endswith("_date")]
date_cols_mcr = [c for c in df.columns if c.startswith("mcr_chow_") and c.endswith("_date")]

if not date_cols_nhc:
    raise ValueError("No NHC date columns found (expected nh_compare_chow_*_date).")
if not date_cols_mcr:
    raise ValueError("No MCR date columns found (expected mcr_chow_*_date).")

# Filter to the 1x1 validation sample
df_11 = df[(df["n_chow_nh_compare"] == 1) & (df["n_chow_mcr"] == 1)].copy()
print(f"\nValidated 1x1 facilities: {len(df_11):,}")

# Get event date (use NHC date as your event month)
df_11["nhc_date"] = df_11.apply(lambda r: first_valid_date(r, date_cols_nhc), axis=1)
df_11["mcr_date"] = df_11.apply(lambda r: first_valid_date(r, date_cols_mcr), axis=1)

# If you want to sanity check disagreements, print a quick summary:
df_11["date_gap_days"] = (df_11["nhc_date"] - df_11["mcr_date"]).dt.days
print("\nDate gap (NHC - MCR) in days, summary:")
print(df_11["date_gap_days"].describe())

# Study window: Jan 2017 to Jun 2024 (inclusive)
start = pd.Timestamp("2017-01-01")
end   = pd.Timestamp("2024-06-30")

df_11_win = df_11[df_11["nhc_date"].between(start, end)].copy()
print(f"\nValidated 1x1 changes within study window (by NHC date): {len(df_11_win):,}")

# -----------------------------
# Bar chart by MONTH (talk-friendly if not too busy)
# -----------------------------
df_11_win["year_month"] = df_11_win["nhc_date"].dt.to_period("M").astype(str)
monthly = df_11_win.groupby("year_month").size()

# Ensure full sequence of months from start to end
all_months = pd.period_range(start=start, end=end, freq="M").astype(str)
monthly = monthly.reindex(all_months, fill_value=0)

plt.figure(figsize=(10,4))
plt.bar(monthly.index, monthly.values)
plt.xticks(rotation=90, fontsize=7)
plt.ylabel("# validated ownership changes")
plt.title("Validated Ownership Changes by Month (NHC=1 and HCRIS/MCR=1)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "validated_changes_by_month.png"), dpi=200)
plt.close()

# -----------------------------
# Bar chart by YEAR (cleanest for a talk)
# -----------------------------
df_11_win["year"] = df_11_win["nhc_date"].dt.year
yearly = df_11_win.groupby("year").size()

# Force full year coverage
all_years = list(range(2017, 2025))  # 2017..2024
yearly = yearly.reindex(all_years, fill_value=0)

plt.figure()
plt.bar(yearly.index, yearly.values)
plt.xlabel("Year")
plt.ylabel("# validated ownership changes")
plt.title("Validated Ownership Changes by Year (NHC=1 and HCRIS/MCR=1)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "validated_changes_by_year.png"), dpi=200)
plt.close()

print("\n=== Yearly validated change counts (NHC=1 & MCR=1, within window) ===")
print(yearly.to_string())

print(f"\nSaved figures to: {os.path.abspath(OUT_DIR)}")
print("Done.")


=== Cross-tab (0/1/2+) ===
mcr_bin     0     1   2+
nhc_bin                 
0        6288   499   15
1        1874  1759  101
2+       1137  1377  535

Validated 1x1 facilities: 1,759

Date gap (NHC - MCR) in days, summary:
count    1759.000000
mean      -25.674815
std       277.140900
min     -2132.000000
25%         0.000000
50%         0.000000
75%         0.000000
max      2403.000000
Name: date_gap_days, dtype: float64

Validated 1x1 changes within study window (by NHC date): 1,737

=== Yearly validated change counts (NHC=1 & MCR=1, within window) ===
year
2017    249
2018    266
2019    279
2020    187
2021    282
2022    211
2023    206
2024     57

Saved figures to: C:\Repositories\white-bowblis-nhmc\outputs\plots
Done.
